In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q chromadb
!pip install -q langchain.text_splitters
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes

In [ ]:
college_text = """
The college library is open from 8 AM to 8 PM.

Minimum attendance required is 75%.

The placement office is located in Block B.

Computer Science department offers AI and ML specialization.

Students can apply for internships through the placement portal.

Semester exams are conducted twice a year.

The college cafeteria operates from 9 AM to 6 PM.
"""

with open("college_info.txt", "w") as f:
    f.write(college_text)

print("Dataset created successfully!")

Dataset created successfully!


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load document
loader = TextLoader("college_info.txt")
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Number of chunks:", len(docs))

Number of chunks: 1


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded!")

/tmp/ipykernel_18955/1776687201.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded!


In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding
)

print("Vector database created!")

Vector database created!


In [ ]:
from transformers import pipeline

llm_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=100
)

print("LLM Loaded Successfully!")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

LLM Loaded Successfully!


In [ ]:
def chatbot(query):

    # Retrieve relevant documents
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    retrieved_docs = retriever.invoke(query)

    context = ""

    for doc in retrieved_docs:
        context += doc.page_content + "\n"

    # Create prompt
    prompt = f"""
    Answer the question based only on the context below.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # Generate answer
    response = llm_pipeline(prompt)

    return response[0]['generated_text']

In [ ]:
query = "What is the attendance requirement?"

answer = chatbot(query)

print("Question:", query)
print("Answer:", answer)

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the attendance requirement?
Answer: 
    Answer the question based only on the context below.

    Context:
    The college library is open from 8 AM to 8 PM.

Minimum attendance required is 75%.

The placement office is located in Block B.

Computer Science department offers AI and ML specialization.

Students can apply for internships through the placement portal.

Semester exams are conducted twice a year.

The college cafeteria operates from 9 AM to 6 PM.


    Question:
    What is the attendance requirement?

    Answer:
    


In [ ]:
while True:

    query = input("Ask a question: ")

    if query.lower() == "exit":
        break

    answer = chatbot(query)

    print("\nAnswer:", answer)
    print("-" * 50)

Ask a question: What is the attendance requirement?


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: 
    Answer the question based only on the context below.

    Context:
    The college library is open from 8 AM to 8 PM.

Minimum attendance required is 75%.

The placement office is located in Block B.

Computer Science department offers AI and ML specialization.

Students can apply for internships through the placement portal.

Semester exams are conducted twice a year.

The college cafeteria operates from 9 AM to 6 PM.


    Question:
    What is the attendance requirement?

    Answer:
    
--------------------------------------------------
Ask a question: exit
